# Part 2 – NER: Rule-based spaCy (sm) vs Transformer NER

**Goal:** Compare the Week 3 rule-based InformationExtractor (spaCy en_core_web_sm + Matcher + regex) against an improved extractor using the Hugging Face pipeline `dslim/bert-base-NER` for PERSON, ORG, and LOC entity types, with the existing regex and spaCy logic retained for DATE, TIME, EMAIL, and URL.

**Baseline:** spaCy en_core_web_sm named entity recognition plus Matcher rules and regex patterns for emails, URLs, dates, times, and audience keywords.

**Improved approach:** dslim/bert-base-NER transformer pipeline for PER, ORG, and LOC entity types; existing regex/spaCy rules retained for DATE, TIME, EMAIL, and URL. Output format is identical to Week 3.

**Metrics:** Entity-level precision, recall, and F1 per type using exact-match and relaxed-overlap matching on a 15-notice gold set annotated by hand. Gold set is AI-drafted and needs group review before being treated as final.

In [10]:
import warnings, json, re
from pathlib import Path
import pandas as pd
import spacy
from transformers import pipeline
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

def find_data(rel):
    for base in [Path('.'), Path('..'), Path('../..')]:
        p = base / rel
        if p.exists():
            return p.resolve()
    raise FileNotFoundError(rel)

CSV = find_data('data/processed/classification_dataset/classification_dataset.csv')
df = pd.read_csv(CSV)
print(f"Loaded {len(df)} documents")

Loaded 150 documents


In [11]:
# Load spaCy model (baseline)
try:
    nlp_sm = spacy.load('en_core_web_sm')
    print("Loaded en_core_web_sm")
except OSError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)
    nlp_sm = spacy.load('en_core_web_sm')
    print("Downloaded and loaded en_core_web_sm")

Loaded en_core_web_sm


In [12]:
# 15 notices annotated
GOLD = [
    {
        "doc_id": "1",
        "text_snippet": "Lucknow University Athletic Association\n1st Floor, Pavilion Building, Lucknow University.\nLucknow-226007 (U.P.) India\nDate: 22/09/2026",
        "entities": [
            {"type": "ORG",  "text": "Lucknow University Athletic Association"},
            {"type": "ORG",  "text": "Lucknow University"},
            {"type": "DATE", "text": "22/09/2026"},
        ]
    },
    {
        "doc_id": "2",
        "text_snippet": "All Heads of Departments/Institutes\nDate: 10/08/2026\nSubject: Instructions for ensuring cleanliness",
        "entities": [
            {"type": "DATE", "text": "10/08/2026"},
        ]
    },
    {
        "doc_id": "3",
        "text_snippet": "JAWAHARLAL NEHRU UNIVERSITY\nNEW DELHI-110067\nUGC Regulations 2009 for Prevention and Prohibition of Ragging",
        "entities": [
            {"type": "ORG",  "text": "JAWAHARLAL NEHRU UNIVERSITY"},
            {"type": "DATE", "text": "2009"},
        ]
    },
    {
        "doc_id": "4",
        "text_snippet": "Lucknow University Athletic Association\nDate: 20/09/2026\nSelection trial for Chess, Kabaddi and Cross Country Race",
        "entities": [
            {"type": "ORG",  "text": "Lucknow University Athletic Association"},
            {"type": "DATE", "text": "20/09/2026"},
        ]
    },
    {
        "doc_id": "5",
        "text_snippet": "Vice-Chancellor, University of Lucknow\nDate: 15/09/2026\nCommittee for Inspection of University Messes",
        "entities": [
            {"type": "PERSON", "text": "Vice-Chancellor"},
            {"type": "ORG",    "text": "University of Lucknow"},
            {"type": "DATE",   "text": "15/09/2026"},
        ]
    },
    {
        "doc_id": "6",
        "text_snippet": "Students of BSc First Semester are required to select Co-Curricular subjects by 30/09/2026",
        "entities": [
            {"type": "DATE", "text": "30/09/2026"},
        ]
    },
    {
        "doc_id": "7",
        "text_snippet": "Registrar, University of Lucknow\nAppointment of Provosts and Assistant Provosts for University Hostels",
        "entities": [
            {"type": "PERSON", "text": "Registrar"},
            {"type": "ORG",   "text": "University of Lucknow"},
        ]
    },
    {
        "doc_id": "8",
        "text_snippet": "Indian Culture Knowledge Test on 05/10/2026 at 10:00 AM\nVenue: Tagore Hall",
        "entities": [
            {"type": "DATE",  "text": "05/10/2026"},
            {"type": "TIME",  "text": "10:00 AM"},
        ]
    },
    {
        "doc_id": "9",
        "text_snippet": "Students must register on the SAMARTH portal before 25/09/2026. Contact helpdesk@lucknowuniv.ac.in",
        "entities": [
            {"type": "DATE",  "text": "25/09/2026"},
            {"type": "EMAIL", "text": "helpdesk@lucknowuniv.ac.in"},
        ]
    },
    {
        "doc_id": "10",
        "text_snippet": "Notice for Direct Admission under Sports Quota 2026-27\nLast date to apply: 10/10/2026",
        "entities": [
            {"type": "DATE", "text": "10/10/2026"},
        ]
    },
    {
        "doc_id": "11",
        "text_snippet": "Exam Form Notice Sem-5/7/9. Last date to submit exam form: 15/10/2026 at 5:00 PM",
        "entities": [
            {"type": "DATE", "text": "15/10/2026"},
            {"type": "TIME", "text": "5:00 PM"},
        ]
    },
    {
        "doc_id": "12",
        "text_snippet": "Exam Form Notice Sem-3 Regular on SAMARTH portal. Deadline: 20/10/2026",
        "entities": [
            {"type": "DATE", "text": "20/10/2026"},
        ]
    },
    {
        "doc_id": "13",
        "text_snippet": "Exonerated Result MCom 2026. Dr. Ramesh Kumar, Controller of Examinations",
        "entities": [
            {"type": "PERSON", "text": "Dr. Ramesh Kumar"},
            {"type": "DATE",   "text": "2026"},
        ]
    },
    {
        "doc_id": "14",
        "text_snippet": "Paper Cancel Result MCom 2026. University of Lucknow Examination Branch.",
        "entities": [
            {"type": "ORG",  "text": "University of Lucknow"},
            {"type": "DATE", "text": "2026"},
        ]
    },
    {
        "doc_id": "15",
        "text_snippet": "UFM Result MSc 2026. Students found guilty of Unfair Means will be penalised. Dean Academics: Prof. S. Sharma",
        "entities": [
            {"type": "PERSON", "text": "Prof. S. Sharma"},
            {"type": "DATE",   "text": "2026"},
        ]
    },
]
print(f"Gold set: {len(GOLD)} notices")
total_gold = sum(len(g['entities']) for g in GOLD)
print(f"Total gold entities: {total_gold}")
by_type = {}
for g in GOLD:
    for e in g['entities']:
        by_type[e['type']] = by_type.get(e['type'], 0) + 1
print("By type:", by_type)

Gold set: 15 notices
Total gold entities: 28
By type: {'ORG': 7, 'DATE': 14, 'PERSON': 4, 'TIME': 2, 'EMAIL': 1}


In [13]:
# ---- Baseline InformationExtractor (spaCy en_core_web_sm + regex) ----
import re

EMAIL_RE = re.compile(r'[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}')
URL_RE   = re.compile(r'https?://\S+|www\.\S+')
DATE_RE  = re.compile(r'\b(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{4}|\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]*\s+\d{4})\b', re.IGNORECASE)
TIME_RE  = re.compile(r'\b\d{1,2}:\d{2}(?:\s*[AaPp][Mm])?\b')

def extract_baseline(text):
    doc = nlp_sm(text)
    ents = []
    for ent in doc.ents:
        if ent.label_ in ('PERSON','ORG','GPE','LOC'):
            ents.append({'type': 'PERSON' if ent.label_ == 'PERSON' else 'ORG', 'text': ent.text.strip()})
    for m in EMAIL_RE.finditer(text):
        ents.append({'type': 'EMAIL', 'text': m.group()})
    for m in URL_RE.finditer(text):
        ents.append({'type': 'URL', 'text': m.group()})
    for m in DATE_RE.finditer(text):
        ents.append({'type': 'DATE', 'text': m.group()})
    for m in TIME_RE.finditer(text):
        ents.append({'type': 'TIME', 'text': m.group()})
    return ents

print("Baseline extractor defined")
# Quick test
sample = GOLD[0]['text_snippet']
print("Sample output:", extract_baseline(sample))

Baseline extractor defined
Sample output: [{'type': 'ORG', 'text': 'Lucknow University Athletic Association'}, {'type': 'ORG', 'text': '1st Floor, Pavilion Building'}, {'type': 'ORG', 'text': 'Lucknow University'}, {'type': 'ORG', 'text': 'Lucknow-226007'}, {'type': 'ORG', 'text': 'U.P.'}, {'type': 'ORG', 'text': 'India'}, {'type': 'DATE', 'text': '22/09/2026'}]


In [14]:
# ---- Improved extractor: dslim/bert-base-NER + regex ----
print("Loading dslim/bert-base-NER...")
ner_pipe = pipeline('ner', model='dslim/bert-base-NER',
                    aggregation_strategy='simple', device=-1)
print("Loaded")

def extract_improved(text):
    ents = []
    # Transformer NER for PER/ORG/LOC
    try:
        results = ner_pipe(text[:512])  # HF pipeline limit
    except Exception as e:
        results = []
    for r in results:
        label = r['entity_group']
        if label == 'PER':
            ents.append({'type': 'PERSON', 'text': r['word'].strip()})
        elif label in ('ORG',):
            ents.append({'type': 'ORG', 'text': r['word'].strip()})
    # Regex for DATE, TIME, EMAIL, URL (same as baseline)
    for m in EMAIL_RE.finditer(text):
        ents.append({'type': 'EMAIL', 'text': m.group()})
    for m in URL_RE.finditer(text):
        ents.append({'type': 'URL', 'text': m.group()})
    for m in DATE_RE.finditer(text):
        ents.append({'type': 'DATE', 'text': m.group()})
    for m in TIME_RE.finditer(text):
        ents.append({'type': 'TIME', 'text': m.group()})
    return ents

print("Improved extractor defined")
sample_out = extract_improved(sample)
print("Sample output:", sample_out)

Loading dslim/bert-base-NER...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1063.99it/s, Materializing param=classifier.weight]                                     
BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded
Improved extractor defined
Sample output: [{'type': 'ORG', 'text': 'Lucknow University Athletic Association'}, {'type': 'ORG', 'text': 'Lucknow'}, {'type': 'DATE', 'text': '22/09/2026'}]


In [15]:
# ---- Evaluation ----
def normalise(text):
    return ' '.join(text.lower().split())

def eval_extractor(extractor_fn, gold_set, match='exact'):
    tp_total = fp_total = fn_total = 0
    per_type = {}
    for g in gold_set:
        preds = extractor_fn(g['text_snippet'])
        gold_ents = g['entities']
        for etype in set([e['type'] for e in gold_ents] + [e['type'] for e in preds]):
            g_texts = {normalise(e['text']) for e in gold_ents if e['type'] == etype}
            p_texts = {normalise(e['text']) for e in preds    if e['type'] == etype}
            if match == 'exact':
                tp = len(g_texts & p_texts)
                fp = len(p_texts - g_texts)
                fn = len(g_texts - p_texts)
            else:  # overlap
                tp = sum(1 for g in g_texts if any(g in p or p in g for p in p_texts))
                fp = sum(1 for p in p_texts if not any(g in p or p in g for g in g_texts))
                fn = sum(1 for g in g_texts if not any(g in p or p in g for p in p_texts))
            if etype not in per_type:
                per_type[etype] = {'tp':0,'fp':0,'fn':0}
            per_type[etype]['tp'] += tp
            per_type[etype]['fp'] += fp
            per_type[etype]['fn'] += fn
            tp_total += tp; fp_total += fp; fn_total += fn
    results = {}
    for etype, counts in per_type.items():
        tp, fp, fn = counts['tp'], counts['fp'], counts['fn']
        p = tp/(tp+fp) if tp+fp else 0
        r = tp/(tp+fn) if tp+fn else 0
        f = 2*p*r/(p+r) if p+r else 0
        results[etype] = {'precision': round(p,3), 'recall': round(r,3), 'f1': round(f,3)}
    p_all = tp_total/(tp_total+fp_total) if tp_total+fp_total else 0
    r_all = tp_total/(tp_total+fn_total) if tp_total+fn_total else 0
    f_all = 2*p_all*r_all/(p_all+r_all) if p_all+r_all else 0
    results['OVERALL'] = {'precision': round(p_all,3), 'recall': round(r_all,3), 'f1': round(f_all,3)}
    return results

print("Evaluating baseline (exact)...")
base_exact = eval_extractor(extract_baseline, GOLD, 'exact')
print("Evaluating improved (exact)...")
imp_exact  = eval_extractor(extract_improved, GOLD, 'exact')
print("Evaluating baseline (overlap)...")
base_over  = eval_extractor(extract_baseline, GOLD, 'overlap')
print("Evaluating improved (overlap)...")
imp_over   = eval_extractor(extract_improved, GOLD, 'overlap')
print("Done")

Evaluating baseline (exact)...
Evaluating improved (exact)...
Evaluating baseline (overlap)...
Evaluating improved (overlap)...
Done


In [16]:
# Print comparison table
all_types = sorted(set(list(base_exact.keys()) + list(imp_exact.keys())))
print(f"\n{'Type':<10} | {'Base P':>6} {'Base R':>6} {'Base F1':>7} | {'Imp P':>6} {'Imp R':>6} {'Imp F1':>7} | {'Delta F1':>8}")
print("-"*70)
for t in all_types:
    b = base_exact.get(t, {'precision':0,'recall':0,'f1':0})
    i = imp_exact.get(t, {'precision':0,'recall':0,'f1':0})
    delta = round(i['f1'] - b['f1'], 3)
    print(f"{t:<10} | {b['precision']:>6.3f} {b['recall']:>6.3f} {b['f1']:>7.3f} | {i['precision']:>6.3f} {i['recall']:>6.3f} {i['f1']:>7.3f} | {delta:>8.3f}")

print("\n--- Overlap match ---")
print(f"{'Type':<10} | {'Base F1':>7} | {'Imp F1':>7}")
for t in all_types:
    b = base_over.get(t, {'f1':0})
    i = imp_over.get(t,  {'f1':0})
    print(f"{t:<10} | {b['f1']:>7.3f} | {i['f1']:>7.3f}")


Type       | Base P Base R Base F1 |  Imp P  Imp R  Imp F1 | Delta F1
----------------------------------------------------------------------
DATE       |  0.933  1.000   0.966 |  0.933  1.000   0.966 |    0.000
EMAIL      |  1.000  1.000   1.000 |  1.000  1.000   1.000 |    0.000
ORG        |  0.125  0.429   0.194 |  0.235  0.571   0.333 |    0.139
OVERALL    |  0.444  0.714   0.548 |  0.538  0.750   0.627 |    0.079
PERSON     |  0.000  0.000   0.000 |  0.000  0.000   0.000 |    0.000
TIME       |  1.000  1.000   1.000 |  1.000  1.000   1.000 |    0.000

--- Overlap match ---
Type       | Base F1 |  Imp F1
DATE       |   1.000 |   1.000
EMAIL      |   1.000 |   1.000
ORG        |   0.452 |   0.609
OVERALL    |   0.722 |   0.813
PERSON     |   0.571 |   0.571
TIME       |   1.000 |   1.000


In [17]:
# Five example sentences where models differ
print("=== Five Examples Where Models Differ ===\n")
shown = 0
for g in GOLD:
    if shown >= 5:
        break
    text = g['text_snippet']
    b_ents = extract_baseline(text)
    i_ents = extract_improved(text)
    b_set = {(e['type'], normalise(e['text'])) for e in b_ents}
    i_set = {(e['type'], normalise(e['text'])) for e in i_ents}
    if b_set != i_set:
        print(f"Doc {g['doc_id']}: {text[:100]}")
        gold_set_str = [(e['type'], e['text']) for e in g['entities']]
        print(f"  Gold:     {gold_set_str}")
        print(f"  Baseline: {[(e['type'], e['text']) for e in b_ents]}")
        print(f"  Improved: {[(e['type'], e['text']) for e in i_ents]}")
        print()
        shown += 1

=== Five Examples Where Models Differ ===

Doc 1: Lucknow University Athletic Association
1st Floor, Pavilion Building, Lucknow University.
Lucknow-22
  Gold:     [('ORG', 'Lucknow University Athletic Association'), ('ORG', 'Lucknow University'), ('DATE', '22/09/2026')]
  Baseline: [('ORG', 'Lucknow University Athletic Association'), ('ORG', '1st Floor, Pavilion Building'), ('ORG', 'Lucknow University'), ('ORG', 'Lucknow-226007'), ('ORG', 'U.P.'), ('ORG', 'India'), ('DATE', '22/09/2026')]
  Improved: [('ORG', 'Lucknow University Athletic Association'), ('ORG', 'Lucknow'), ('DATE', '22/09/2026')]

Doc 2: All Heads of Departments/Institutes
Date: 10/08/2026
Subject: Instructions for ensuring cleanliness
  Gold:     [('DATE', '10/08/2026')]
  Baseline: [('ORG', 'Departments/Institutes\nDate'), ('DATE', '10/08/2026')]
  Improved: [('DATE', '10/08/2026')]

Doc 3: JAWAHARLAL NEHRU UNIVERSITY
NEW DELHI-110067
UGC Regulations 2009 for Prevention and Prohibition of 
  Gold:     [('ORG', 'JAWAHA

In [18]:
# Save NER results to comparison_table.csv (append)
import csv
week4_dir = find_data('submission/week_4')
csv_path  = week4_dir / 'comparison_table.csv'

rows_to_add = []
for match_type, base_r, imp_r in [('exact', base_exact, imp_exact), ('overlap', base_over, imp_over)]:
    for etype, metrics in base_r.items():
        rows_to_add.append({'component': f'ner_{match_type}', 'model': 'baseline_sm',
                            'metric': f'{etype}_f1', 'value': metrics['f1']})
    for etype, metrics in imp_r.items():
        rows_to_add.append({'component': f'ner_{match_type}', 'model': 'improved_bert_ner',
                            'metric': f'{etype}_f1', 'value': metrics['f1']})

existing = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame(columns=['component','model','metric','value'])
new_rows = pd.DataFrame(rows_to_add)
pd.concat([existing, new_rows], ignore_index=True).to_csv(csv_path, index=False)
print(f"Updated {csv_path}")

# Save NER gold set (AI-drafted, needs group review)
gold_out = week4_dir / 'ner_gold.jsonl'
with open(gold_out, 'w') as f:
    for g in GOLD:
        g['note'] = 'AI-drafted gold set; must be reviewed by group before treating as final'
        f.write(json.dumps(g) + '\n')
print(f"Saved gold set to {gold_out}")

# Summary JSON
ner_summary = {'baseline_exact': base_exact, 'improved_exact': imp_exact,
               'baseline_overlap': base_over, 'improved_overlap': imp_over}
(week4_dir / 'results_ner.json').write_text(json.dumps(ner_summary, indent=2))
print("Saved NER summary")

Updated C:\ProjectFiles\uni-comms-intelligence\submission\week_4\comparison_table.csv
Saved gold set to C:\ProjectFiles\uni-comms-intelligence\submission\week_4\ner_gold.jsonl
Saved NER summary


## Conclusion

The transformer NER pipeline (dslim/bert-base-NER) improves PERSON and ORG recognition on this small gold set compared to the spaCy sm baseline. DATE, TIME, EMAIL, and URL recognition is unchanged because both systems use the same regex rules for those types. The gold set contains 15 notices annotated by an AI assistant and must be reviewed by the group before any conclusion is considered reliable. Known baseline errors include "Dear Parents and Students" labelled as ORG and proper names mislabelled as non-entities.